# Clase 149 — LoRA / QLoRA: fine-tuning eficiente

Implementamos LoRA desde scratch en numpy (CPU-friendly). La API real con HuggingFace + PEFT requiere GPU y se muestra en markdown.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

# Toy: matriz base W (frozen) 100x100
d_in, d_out = 100, 100
W = rng.standard_normal((d_in, d_out)) * 0.1
print('W shape:', W.shape, '| params:', W.size)

## 1. LoRA desde scratch: `W' = W + B·A`

- `A`: (r, d_out), init normal
- `B`: (d_in, r), init zeros → al inicio, `B·A = 0`, no perturba a W.

In [ ]:
r = 4   # rank
alpha = 8
scale = alpha / r

A = rng.standard_normal((r, d_out)) * 0.01
B = np.zeros((d_in, r))

trainable = A.size + B.size
full = W.size
print(f'trainable (LoRA): {trainable:,} | full FT: {full:,} | ratio: {trainable/full:.2%}')

## 2. Target task: aproximar W_target = W + ΔW con LoRA

Generamos un "ΔW objetivo" low-rank y entrenamos `B·A` para aprenderlo.

In [ ]:
# Target delta: low-rank ground truth
B_true = rng.standard_normal((d_in, r)) * 0.3
A_true = rng.standard_normal((r, d_out)) * 0.3
delta_target = B_true @ A_true
W_target = W + delta_target

# Datos: y = X @ W_target
n = 256
X = rng.standard_normal((n, d_in))
Y = X @ W_target

In [ ]:
# Entreno SOLO A, B (W frozen). Forward: y_hat = X @ (W + scale*B@A)
lr = 0.01
losses = []
for epoch in range(300):
    delta = scale * (B @ A)
    Y_hat = X @ (W + delta)
    err = Y_hat - Y
    loss = (err**2).mean()
    losses.append(loss)
    # Gradientes (regla de cadena)
    grad_delta = 2 * X.T @ err / n   # (d_in, d_out)
    grad_B = scale * grad_delta @ A.T
    grad_A = scale * B.T @ grad_delta
    B -= lr * grad_B
    A -= lr * grad_A

print(f'loss inicial: {losses[0]:.4f} | final: {losses[-1]:.6f}')
print(f'||B@A - B_true@A_true||: {np.linalg.norm(scale*B@A - delta_target):.4f}')

## 3. Inspección: trainable params vs full fine-tuning

In [ ]:
for r_test in [1, 4, 8, 16, 32, 64]:
    tp = r_test * (d_in + d_out)
    print(f'  r={r_test:3d} | trainable={tp:6,} | ratio={tp/full:.2%}')

## 4. QLoRA conceptual: cuantización NF4

QLoRA congela W en 4-bit (NF4 = NormalFloat) y mantiene LoRA en fp16.

**Ahorro de memoria sobre Llama 7B:**

In [ ]:
n_params = 7e9
fp16_gb = n_params * 2 / 1e9          # 14 GB
nf4_gb  = n_params * 0.5 / 1e9        # 3.5 GB (4 bits = 0.5 bytes)
lora_params = 2 * 16 * 4096 * 32      # r=16 sobre ~32 attn layers de 4096
lora_gb = lora_params * 2 / 1e9

print(f'fp16 full:      {fp16_gb:5.2f} GB')
print(f'NF4 base:       {nf4_gb:5.2f} GB')
print(f'LoRA adapters:  {lora_gb:5.4f} GB')
print(f'QLoRA total:    {nf4_gb + lora_gb:5.2f} GB → cabe en GPU consumer (8-12 GB)')

## 5. API real (HuggingFace + PEFT) — requiere GPU

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype='float16')
model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1',
                                              quantization_config=bnb, device_map='auto')
lora = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','k_proj','v_proj','o_proj'],
                   lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # → ~0.5%
# train con SFTTrainer ...
model = model.merge_and_unload()     # merge para deploy
```

## Ejercicio guiado

1. Variar `r` ∈ {2, 4, 8, 16} y reportar loss final + # params.
2. Probar con W_target NO low-rank (full rank random): ¿cuánto puede aproximar LoRA?
3. Implementar `alpha` scheduling (subir scale gradualmente).
4. Bonus: agregar dropout sobre A·B en forward.

## Conclusiones

- LoRA reduce params trainables ~100-1000x sin pérdida de calidad cuando ΔW es low-rank.
- QLoRA permite fine-tuning de 7B en GPU consumer (8 GB).
- `merge_and_unload()` devuelve modelo normal para deploy sin latencia extra.